# Домашнее задание №5. Кластеризация и SVM

**На проверку необходимо приложить следующие файлы:**
  - заполненный ноутбук в формате `.ipynb`.

-------------------------

## Основная часть. (10 баллов)

В этом задании рассматриваются методы кластеризации и метод опорных векторов. В ходе решения вам предстоит:
1. Реализовать K-means и инициализацию K-means++.
2. Применить DBSCAN для поиска зон с высокой плотностью посадки такси.
3. Найти разделяющую прямую SVM с помощью решения двойственной задачи.
4. Реализовать линейный SVM и SVM с RBF-ядром.

#### Полезные ссылки

1. [Описание](https://scikit-learn.org/stable/modules/clustering.html#k-means) K-means от scikit-learn
2. Статья по алгоритму [K-means++](https://theory.stanford.edu/~sergei/papers/kMeansPP-soda.pdf)
3. Статья по алгоритму [DBSCAN](https://cdn.aaai.org/KDD/1996/KDD96-037.pdf)
4. Статья по алгоритму [SVM](https://link.springer.com/content/pdf/10.1007/bf00994018.pdf)
5. [Описание](https://scikit-learn.org/stable/auto_examples/svm/plot_rbf_parameters.html) параметров Radial Basis Function (RBF) от scikit-learn

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import adjusted_rand_score, accuracy_score
from sklearn.datasets import load_iris, make_classification, make_moons
from cvxopt import matrix, solvers

### Кластеризация. (4 балла)

#### Задание 1 (2.5 балла): Реализация K-means

##### Задача 1.1 (1.5 балла)

Реализуйте алгоритм K-means без использования библиотек с реализацией данного метода (например, scikit-learn).

In [ ]:
def k_means(X, k, max_iters=3, tol=1e-4):
    """ Реализуйте алгоритм K-Means """
    np.random.seed(42)
    # Your code here
    pass

# Automatic Checker
def check_k_means():
    iris = load_iris()
    X = iris.data
    y_true = iris.target

    labels_custom, centroids_custom = k_means(X, k=3)
    kmeans_sklearn = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)
    labels_sklearn = kmeans_sklearn.labels_

    score_custom = adjusted_rand_score(y_true, labels_custom)
    score_sklearn = adjusted_rand_score(y_true, labels_sklearn)
    print(score_custom)
    print(score_sklearn)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.scatter(X[:, 0], X[:, 1], c=labels_custom, cmap='viridis', marker='o')
    plt.scatter(centroids_custom[:, 0], centroids_custom[:, 1], s=300, c='red', marker='X')
    plt.title('K-Means Clustering')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')

    assert labels_custom is not None, "Function should return labels."
    assert len(labels_custom) == len(X), "Should return labels for all points."
    assert score_custom > 0.7, "Clustering should achieve a higher score."
    print(f"Task 1.1 Passed! Adjusted Rand Score: {score_custom:.2f}")

In [ ]:
check_k_means()

##### Задача 1.2 (1 балл)

Реализуйте инициализацию K-means++ и используйте её в алгоритме K-means без использования библиотек с реализацией данного метода.

In [ ]:
def initialize_plus_plus(X, k):
    """ Инициализируйте центры кластеров """
    np.random.seed(42)
    # Your code here
    pass

def k_means_plus_plus(X, k, max_iters=100, tol=1e-4):
    """ Реализуйте алгоритм K-means++ """
    centroids = initialize_plus_plus(X, k)
    # Your code here
    pass

# Automatic Checker
def check_k_means_plus_plus():
    iris = load_iris()
    X = iris.data
    y_true = iris.target

    labels_custom, centroids_custom = k_means_plus_plus(X, k=3)
    kmeans_sklearn = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10).fit(X)
    labels_sklearn = kmeans_sklearn.labels_

    score_custom = adjusted_rand_score(y_true, labels_custom)
    score_sklearn = adjusted_rand_score(y_true, labels_sklearn)
    print(score_custom)
    print(score_sklearn)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.scatter(X[:, 0], X[:, 1], c=labels_custom, cmap='viridis', marker='o')
    plt.scatter(centroids_custom[:, 0], centroids_custom[:, 1], s=300, c='red', marker='X')
    plt.title('K-Means++ Clustering')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')

    assert labels_custom is not None, "Function should return labels."
    assert len(labels_custom) == len(X), "Should return labels for all points."
    assert score_custom > 0.7, "Clustering should achieve a higher score."
    print(f"Task 1.2 Passed! Adjusted Rand Score: {score_custom:.2f}")

In [ ]:
check_k_means_plus_plus()

#### Задание 2 (1.5 балла): DBSCAN

Предположим, что вы анализируете движение такси в городе, чтобы определить популярные зоны посадки. У вас есть набор данных, представляющий места посадки такси.

```
data_points = [
    (1.2, 3.1), (2.5, 3.3), (1.8, 2.9), (8.2, 7.5),
    (8.1, 7.7), (25.3, 80.2), (2.0, 3.0), (3.0, 3.5),
    (1.1, 1.2), (90.0, 91.2), (1.5, 3.2), (8.0, 7.6),
    (2.2, 3.1), (7.9, 7.8), (1.3, 3.0)
]
```

Каждая точка представляет место посадки в такси с координатами (x, y) км от центра города.

С помощью алгоритма DBSCAN сгруппируйте эти поездки на такси, чтобы определить зоны с высокой плотностью посадки. Укажите, какие из точек core, а какие являются выбросами.

Указание: выберите максимальное расстояние между точками равным 1.5 и минимальное число точек в группе равным 3.

Для решения этой задачи не используйте библиотеки с реализацией DBSCAN.

In [ ]:
# Your code here

### Метод опорных векторов. (6 баллов)

#### Задание 3 (2 балла): Разделяющая прямая SVM

Рассмотрим набор данных в 2D-пространстве, где первые два значения - это координаты точек, а третье значение - метка класса:

```
data_points = [
    (-4, 2, 1), (-1, -1, -1), (0, -2, -1), (1, -2, -1),
    (2, 3, 1), (3, 3, 1), (4, 4, 1), (-4, -3, -1), (-5, -2, -1),
    (5, 2, 1), (6, 1, 1), (-6, -1, -1), (-7, -3, -1), (7, 3, 1),
    (8, 4, 1), (-8, -5, -1), (9, 5, 1), (-9, -4, -1), (10, 6, 1),
]
```

1. Определите уравнение прямой, разделяющей два класса с максимальным зазором.
2. Определите величину зазора (margin) и вектор, ортогональный разделяющей границе. Под зазором понимается полная ширина полосы между двумя опорными прямыми.
3. Определите, какие точки являются опорными векторами.
4. Постройте график. Выделите опорные векторы.

Указание: используйте постановку SVM для линейно разделимой выборки с жёстким зазором. В данной задаче необходимо преобразовать двойственную задачу SVM к стандартному виду квадратичной задачи и использовать солвер (см. импорты и изучите [документацию cvxopt](https://cvxopt.org/examples/tutorial/qp.html)), чтобы восстановить разделяющую гиперплоскость и опорные вектора.

Солвер принимает квадратичную задачу в следующем виде:
$$
\begin{aligned}
\min_{z} ~~&~~ \frac{1}{2} z^\top P z + q^\top z \\
\text{s.t.} ~~&~~ Gz \leq h \\
~~&~~ Az = b.
\end{aligned}
$$

Выразите двойственную задачу SVM в этом виде, определив матрицы и векторы $P, q, G, h, A, b$.

In [ ]:
# Your code here

#### Задание 4 (4 балла): Реализация SVM

##### Задача 4.1 (1.5 балла)

Реализуйте линейный SVM без использования библиотек с реализацией данного метода.

In [ ]:
def linear_svm(X, y, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
    """ Реализуйте Linear SVM """
    np.random.seed(42)
    # Your code here
    pass

def predict_svm(X, weights, bias):
    return np.sign(np.dot(X, weights) - bias)

# Automatic Checker
def check_linear_svm():
    X, y = make_classification(n_samples=1000, n_features=4, n_informative=2, n_redundant=0, random_state=42)
    y = np.where(y == 0, -1, 1)

    X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

    weights, bias = linear_svm(X, y)
    y_pred = predict_svm(X, weights, bias)

    svm_sklearn = SVC(kernel='linear', random_state=42)
    svm_sklearn.fit(X, y)
    y_pred_sklearn = svm_sklearn.predict(X)

    accuracy = accuracy_score(y, y_pred)
    accuracy_sklearn = accuracy_score(y, y_pred_sklearn)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
    for ax, labels, title in zip(
        axes,
        [y, y_pred, y_pred_sklearn],
        ['Истинные классы', f'Своя реализация: accuracy = {accuracy:.3f}',
         f'sklearn: accuracy = {accuracy_sklearn:.3f}'],
    ):
        ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='bwr', vmin=-1, vmax=1, s=15, alpha=0.6)
        errors = labels != y
        if np.any(errors):
            ax.scatter(X[errors, 0], X[errors, 1], facecolors='none', edgecolors='black',
                       s=45, label='Ошибки')
            ax.legend()
        ax.set(title=title, xlabel='$x_1$', ylabel='$x_2$')
    fig.suptitle('SVM: проекция на первые два признака')
    plt.tight_layout()
    plt.show()

    assert accuracy > 0.85, "Accuracy should be higher"
    print(f"Task 4.1 Passed! Accuracy: {accuracy:.2f}")

In [ ]:
check_linear_svm()

##### Задача 4.2 (2.5 балла)

Реализуйте SVM с RBF-ядром без использования библиотек с реализацией данного метода. Функция `kernel_svm` должна возвращать коэффициенты `alpha` и свободный член `bias`.

In [ ]:
def rbf_kernel(X, Y=None, gamma=0.1):
    if Y is None:
        Y = X
    """ Определите RBF kernel """
    # Your code here
    pass

def kernel_svm(X, y, gamma=0.1, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
    """ Реализуйте Kernel SVM """
    np.random.seed(42)
    # Your code here
    pass

def predict_kernel_svm(X, X_train, y_train, alpha, bias, gamma=0.1):
    K = rbf_kernel(X, X_train, gamma)
    return np.sign(np.dot(K, alpha * y_train) + bias)

# Automatic Checker
def check_kernel_svm():
    X, y = make_moons(n_samples=50, noise=0.01, random_state=42)
    y = np.where(y == 0, -1, 1)

    alpha, bias = kernel_svm(X, y)
    y_pred = predict_kernel_svm(X, X, y, alpha, bias)

    svm_sklearn = SVC(kernel='rbf', gamma=0.1, random_state=42)
    svm_sklearn.fit(X, y)
    y_pred_sklearn = svm_sklearn.predict(X)

    accuracy = accuracy_score(y, y_pred)
    accuracy_sklearn = accuracy_score(y, y_pred_sklearn)

    print(accuracy_sklearn)

    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 150),
        np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 150),
    )
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    custom_scores = rbf_kernel(grid, X, gamma=0.1) @ (alpha * y) + bias
    sklearn_scores = svm_sklearn.decision_function(grid)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
    for ax, labels, scores, title in zip(
        axes,
        [y, y_pred, y_pred_sklearn],
        [None, custom_scores, sklearn_scores],
        ['Истинные классы', f'Своя реализация: accuracy = {accuracy:.3f}',
         f'sklearn: accuracy = {accuracy_sklearn:.3f}'],
    ):
        if scores is not None:
            scores = scores.reshape(xx.shape)
            ax.contourf(xx, yy, scores >= 0, levels=[-0.5, 0.5, 1.5], cmap='bwr', alpha=0.15)
            ax.contour(xx, yy, scores, levels=[0], colors='black', linewidths=1)
        ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='bwr', vmin=-1, vmax=1, s=35)
        errors = labels != y
        if np.any(errors):
            ax.scatter(X[errors, 0], X[errors, 1], facecolors='none', edgecolors='black',
                       s=90, label='Ошибки')
            ax.legend()
        ax.set(title=title, xlabel='$x_1$', ylabel='$x_2$')
    fig.suptitle('SVM с RBF-ядром')
    plt.tight_layout()
    plt.show()

    assert accuracy > 0.8, "Accuracy should be higher"
    print(f"Task 4.2 Passed! Accuracy: {accuracy:.2f}")

In [ ]:
check_kernel_svm()